# Step 08b — Final dataset report

> Summarize the curated dataset (final repository and workflow counts). Pure reporting cell — emits dataset statistics + sanity-check metrics.

**Paper section:** Step 1 — Repository selection and workflow extraction (final reporting).

## Position in pipeline
Reporting-only leaf. Reads
`data/8_dataset_enriched.csv` and chains via `data/step8_report.json` —
independent of the ground truth/IRR steps (9a-12). Placed at 08b, right after the step-8 enrichment it summarizes.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/8_dataset_enriched.csv` | CSV (~20 MB; shipped with replication package) | Step 8 (online mining) |
| `data/step8_report.json` | JSON | Step 8 (integrity chain) |
| `data/mining_config.json` | JSON | `filters.mining_cutoff` source |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/step08b_report.json` | JSON (final dataset stats) | Paper artifact |



In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 08b — starting: generate final report...", flush=True)
"""Step 08b: generate final dataset report.

Produces:
- Final statistics report (JSON) — paper-cited dataset numbers.
- Sanity-check metrics (forks/archived should be 0 post step 8).

Pure reporting cell. No transform output; downstream pipeline consumes
data/8_dataset_enriched.csv directly.
"""

import argparse
import json
from pathlib import Path

import polars as pl

DEFAULT_MINING_CONFIG = Path("data/mining_config.json")

def _load_execution_window(report_path: Path) -> dict | None:
    if not report_path.exists():
        return None
    with report_path.open() as f:
        data = json.load(f)
    started = data.get("started_at")
    finished = data.get("finished_at")
    if not started or not finished:
        return None
    return {"started_at": started, "finished_at": finished}

def _load_mining_cutoff(config_path: Path) -> dict | None:
    if not config_path.exists():
        return None
    with config_path.open() as f:
        config = json.load(f)
    cutoff = config.get("filters", {}).get("mining_cutoff")
    if not cutoff or "start" not in cutoff or "end" not in cutoff:
        return None
    return {"start": cutoff["start"], "end": cutoff["end"]}

def generate_report(
    dataset_path: Path,
    report_path: Path,
    prior_report_path: Path,
    pre_api_path: Path | None = None,
    mining_config_path: Path = DEFAULT_MINING_CONFIG,
) -> dict:
    from dataset_utils.integrity import PipelineIntegrityError, assert_input_matches_report

    if not prior_report_path.exists():
        msg = (
            f"step 08b input: prior report not found at {prior_report_path}. "
            "Run step 8 first."
        )
        raise PipelineIntegrityError(msg)

    input_sha = assert_input_matches_report(
        input_path=dataset_path,
        prior_report_path=prior_report_path,
        label="step 08b input",
    )

    print("  reading input CSV (may take a moment for large files)...", flush=True)
    df = pl.read_csv(dataset_path, encoding="utf-8-sig")
    fieldnames = df.columns

    total_rows = len(df)
    unique_repos = df["repository"].n_unique()

    languages = df["repo_language"].value_counts().sort("count", descending=True).head(10)
    languages_top_10 = {row["repo_language"]: row["count"] for row in languages.to_dicts()}

    bot_counts = df.get_column("is_bot").value_counts()
    bot_vs_human = {str(row["is_bot"]): row["count"] for row in bot_counts.to_dicts()}

    change_types = df["git_change_type"].value_counts()
    git_change_types = {row["git_change_type"]: row["count"] for row in change_types.to_dicts()}

    branches = df["repo_branch"].value_counts().sort("count", descending=True).head(10)
    branches_top_10 = {row["repo_branch"]: row["count"] for row in branches.to_dicts()}

    forks = df.filter(pl.col("repo_is_fork") == True).height
    archived = df.filter(pl.col("repo_is_archived") == True).height
    with_triggers = df.filter(
        pl.col("workflow_triggers").is_not_null() & (pl.col("workflow_triggers") != "")
    ).height
    with_runs = df.filter(
        pl.col("workflow_run_count").is_not_null() & (pl.col("workflow_run_count").cast(pl.Utf8) != "")
    ).height

    repos_before_api: int | None = None
    repos_removed_by_api: int | None = None
    if pre_api_path is not None and pre_api_path.exists():
        print("  reading pre-API CSV...", flush=True)
        pre_api_df = pl.read_csv(pre_api_path, encoding="utf-8-sig")
        repos_before_api = pre_api_df["repository"].n_unique()
        repos_removed_by_api = repos_before_api - unique_repos

    execution = _load_execution_window(prior_report_path)
    mining_cutoff = _load_mining_cutoff(mining_config_path)

    report = {
        "total_rows": total_rows,
        "unique_repos": unique_repos,
        "columns": fieldnames,
        "column_count": len(fieldnames),
        "languages_top_10": languages_top_10,
        "bot_vs_human": bot_vs_human,
        "git_change_types": git_change_types,
        "branches_top_10": branches_top_10,
        "forks": forks,
        "archived": archived,
        "workflows_with_triggers": with_triggers,
        "workflows_with_run_count": with_runs,
        "repos_before_api_filtering": repos_before_api,
        "repos_removed_by_api": repos_removed_by_api,
        "mining_cutoff": mining_cutoff,
        "execution": execution,
        "input_file": str(dataset_path),
        "input_sha256": input_sha,
        "prior_report": str(prior_report_path),
    }

    with report_path.open("w") as f:
        json.dump(report, f, indent=2)

    return report

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Generate final dataset report.")
    parser.add_argument(
        "--input",
        type=Path,
        default=Path("data/8_dataset_enriched.csv"),
        help="Input enriched dataset CSV file path.",
    )
    parser.add_argument(
        "--report",
        type=Path,
        default=Path("data/step08b_report.json"),
        help="Output final report JSON file path.",
    )
    parser.add_argument(
        "--prior-report",
        type=Path,
        default=Path("data/step8_report.json"),
        help="Step 8 report JSON used for SHA-chain integrity check.",
    )
    parser.add_argument(
        "--pre-api-input",
        type=Path,
        default=None,
        help="Pre-API filtered dataset (output of step 7) to compute repos removed by API.",
    )
    parser.add_argument(
        "--mining-config",
        type=Path,
        default=DEFAULT_MINING_CONFIG,
        help="Mining config JSON sourced for filters.mining_cutoff.",
    )
    return parser.parse_args()

def main() -> None:
    args = parse_args()
    report = generate_report(
        args.input,
        args.report,
        args.prior_report,
        args.pre_api_input,
        args.mining_config,
    )

    print("Final report generated.")
    print(f"Dataset: {args.input}")
    print(f"Report: {args.report}")
    print("-" * 40)
    print(f"Total rows: {report['total_rows']}")
    print(f"Unique repos: {report['unique_repos']}")
    print(f"Columns: {report['column_count']}")
    print(f"Forks: {report['forks']}")
    print(f"Archived: {report['archived']}")
    print(f"Workflows with triggers: {report['workflows_with_triggers']}")
    print(f"Workflows with run count: {report['workflows_with_run_count']}")
    if report["repos_before_api_filtering"] is not None:
        print(f"Repos before API filtering: {report['repos_before_api_filtering']}")
        print(f"Repos removed by API: {report['repos_removed_by_api']}")

main()


Step 08b — starting: generate final report...
  reading input CSV (may take a moment for large files)...
Final report generated.
Dataset: data/8_dataset_enriched.csv
Report: data/step08b_report.json
----------------------------------------
Total rows: 49836
Unique repos: 18025
Columns: 26
Forks: 0
Archived: 0
Workflows with triggers: 49836
Workflows with run count: 49836


---

**Next step:** [`09_build_ground_truth_candidates.ipynb`](./09_build_ground_truth_candidates.ipynb)
